# Querying Planetary Computer STAC items with rustac[rustac](https://github.com/stac-utils/rustac-py) is a Rust-powered STAC toolkit for Python. The Planetary Computer publishes a GeoParquet snapshot of every collection's STAC items, and rustac reads it through a bundled DuckDB engine. Instead of downloading the whole file and filtering in pandas, you push spatial, temporal, and property filters down to the Parquet and read back only the items that match.This notebook queries the [Sentinel-2 L2A](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a) snapshot over Portland, Oregon. The companion [rustac tutorial](../overview/rustac.md) has the full narrative.> **Note on coverage:** the Sentinel-2 GeoParquet snapshot is a point-in-time export covering 2015-07 through 2018-10, not the live archive. Use it for bulk historical analysis. For current acquisitions, query the live STAC API with `pystac-client`.

## Install

In [ ]:
%pip install --quiet 'rustac[arrow]' pystac-client planetary-computer geopandas pyarrow

## Find the GeoParquet snapshotEvery Planetary Computer collection carries a collection-level `geoparquet-items` asset that points at its STAC-items snapshot. The href resolves to a directory of monthly `part-NNNN` Parquet files on the `pcstacitems` storage account.**Expected result:** `abfs://items/sentinel-2-l2a.parquet`.

In [ ]:
import pystac_client

catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
asset = catalog.get_collection("sentinel-2-l2a").assets["geoparquet-items"]
asset.href

## Authenticate the DuckDB engineThe account name looks public, but reads still need a SAS token. The high-level `rustac.search()` coroutine cannot pass Azure credentials, so use `rustac.DuckdbClient` instead and configure its connection once: fetch a container SAS from the Planetary Computer token API, register an Azure secret, and switch the Azure transport to curl.The `azure_transport_option_type = 'curl'` line is not optional. Without it, DuckDB's default Azure transport fails with an opaque SSL CA-certificate error. The SAS expires after about an hour, so long-running jobs re-fetch it.**Expected result:** a configured `client`, no output printed.

In [ ]:
import json
import urllib.request
import rustac

sas = json.load(urllib.request.urlopen(
    "https://planetarycomputer.microsoft.com/api/sas/v1/token/pcstacitems/items"
))["token"]

client = rustac.DuckdbClient()
client.execute("INSTALL azure; LOAD azure; SET azure_transport_option_type = 'curl';")
client.execute(
    "CREATE SECRET pc (TYPE azure, PROVIDER config, ACCOUNT_NAME 'pcstacitems', "
    f"CONNECTION_STRING 'BlobEndpoint=https://pcstacitems.blob.core.windows.net;SharedAccessSignature={sas}')"
)

## Query the snapshotThe original quickstart read the whole snapshot into a GeoDataFrame and filtered in memory. rustac pushes the same filters into the read, so only matching rows cross the network. The glob (`/*.parquet`) spans the monthly part files.`DuckdbClient.search` is synchronous (no `await`); the bundled engine scans the part files for you, which takes a little time on the first call.**Expected result:** a handful of Portland scenes (5).

In [ ]:
SNAPSHOT = "az://items/sentinel-2-l2a.parquet/*.parquet"

items = client.search(
    SNAPSHOT,
    collections=["sentinel-2-l2a"],
    bbox=[-122.7, 45.5, -122.6, 45.6],
    datetime="2017-07-01/2017-08-01",
)
len(items)

## Filter on space, time, and properties`search()` accepts the STAC query parameters you would expect, including CQL2-JSON property filters. Here we add a cloud-cover threshold.**Expected result:** a list of low-cloud scenes over the wider summer window.

In [ ]:
items = client.search(
    SNAPSHOT,
    collections=["sentinel-2-l2a"],
    bbox=[-122.7, 45.5, -122.6, 45.6],
    datetime="2017-06-01/2017-09-01",
    filter={"op": "<", "args": [{"property": "eo:cloud_cover"}, 20]},
)
len(items)

## Write results without materializing Python objectsFor bulk work, skip the round-trip through Python dictionaries. `search_to_arrow` takes the same arguments as `search` and returns an Arrow table. It is backed by `arro3` (rustac's Arrow runtime), so adopt it into PyArrow with `pa.table(...)` — a zero-copy hand-off — before writing it to a new Parquet file.**Expected result:** a `portland-2017.parquet` file written locally; `table.num_rows` is 5.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.table(client.search_to_arrow(
    SNAPSHOT,
    collections=["sentinel-2-l2a"],
    bbox=[-122.7, 45.5, -122.6, 45.6],
    datetime="2017-07-01/2017-08-01",
))
pq.write_table(table, "portland-2017.parquet")
table.num_rows

## Bridge to GeoPandasTo analyze results as a GeoDataFrame, convert the Arrow table. The snapshot geometries are lon/lat, but the CRS is not carried on the Arrow table, so set it explicitly.**Expected result:** a 5-row GeoDataFrame with `Polygon` geometry.

In [ ]:
import geopandas

gdf = geopandas.GeoDataFrame.from_arrow(table).set_crs(4326)
gdf[["id", "datetime", "eo:cloud_cover"]].head()

## From metadata to pixelsA search returns item metadata, not imagery. To read the actual rasters, pull asset hrefs from the results and sign them, then hand them to a reader such as [async-geotiff](../overview/async-geotiff.md) for windowed reads.**Expected result:** a signed HTTPS URL for the B04 (red) band of the first scene.

In [ ]:
import planetary_computer

href = planetary_computer.sign(items[0]["assets"]["B04"]["href"])
href

## When to use something elseThe live STAC API (through `pystac-client`) is the right tool for small, targeted queries and for data newer than the snapshot. Reach for rustac and the GeoParquet snapshot when you are scanning across many items at once, where pushing filters down and reading only matches saves real time and bandwidth. To ask analytical questions or join the catalog against other datasets, see the [DuckDB tutorial](../overview/duckdb.md), which uses the same authentication.